# Ориентация текста на изображении

Нужно оценить вероятность того, что текстовый бокс перевёрнут на 180°. Итоговая метрика — **1 − Brier Score**.

Я использовал одну CNN **PP-LCNet x1.0** и дообучил её последний классификационный слой. Полученный балл на скрытом тесте: **0,95582654** (Brier = 0,04417346). Ноутбук и файл `text_orientation_finetuned.onnx` должны лежать рядом. После запуска появится `submission.csv`.

In [ ]:
from pathlib import Path
import csv
import hashlib
import math
import time
import numpy as np
from PIL import Image, ImageOps
import onnxruntime as ort

ROOT = Path.cwd()
candidates = [ROOT / "test/test/images", ROOT / "test/images", ROOT / "images"]
image_dir = next((path for path in candidates if path.is_dir()), None)
if image_dir is None:
    raise FileNotFoundError("Не найдена папка с тестовыми PNG")

sample_candidates = [
    ROOT / "test/sample_submission.csv",
    ROOT / "sample_submission.csv",
]
sample_path = next((path for path in sample_candidates if path.exists()), None)
if sample_path is not None:
    with sample_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        image_ids = [row["image_id"] for row in reader]
else:
    image_ids = sorted(path.stem for path in image_dir.glob("*.png"))

assert image_ids and len(image_ids) == len(set(image_ids))
print(f"Найдено {len(image_ids):,} изображений. Папка: {image_dir}")

## Данные и модель

Тестовые изображения уже вырезаны в отдельные боксы. Поэтому детектор текста не нужен: достаточно классификатора, который различает ориентации 0° и 180°. Я взял [PP-LCNet x1.0](https://www.paddleocr.ai/main/en/version3.x/module_usage/textline_orientation_classification.html), заморозил свёрточную часть и дообучил последний слой на открытом наборе PaddleClas (1800 обучающих и 200 проверочных исходных строк). Тестовые изображения в обучении не участвовали.

Для каждого бокса я усредняю два предсказания: исходное изображение и его поворот на 180°. Это делает вероятности согласованными при повороте.

## Веса модели

Дообученная модель сохранена отдельно в файле `text_orientation_finetuned.onnx`. Положите его рядом с ноутбуком. Другие файлы проекта не нужны; кроме модели понадобятся только тестовые PNG и `sample_submission.csv`.

Для запуска установите `numpy`, `Pillow` и `onnxruntime`. Исходная архитектура и веса — [PaddleOCR / RapidOCR](https://github.com/RapidAI/RapidOCR/blob/main/python/rapidocr/default_models.yaml).

## Предобработка

Размер входа CNN — **3 × 80 × 160**. Сохраняю пропорции, свободное место справа заполняю нулями, каналы располагаю в порядке BGR и перевожу значения в диапазон от −1 до 1.

In [ ]:
def prepare(image):
    image = ImageOps.exif_transpose(image).convert("RGB")
    resized_width = min(160, math.ceil(80 * image.width / image.height))
    rgb = np.asarray(
        image.resize((resized_width, 80), Image.Resampling.BILINEAR),
        dtype=np.float32,
    )
    result = np.zeros((3, 80, 160), dtype=np.float32)
    result[:, :, :resized_width] = rgb[:, :, ::-1].transpose(2, 0, 1) / 127.5 - 1
    return result

MODEL_PATH = ROOT / "text_orientation_finetuned.onnx"
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Положите {MODEL_PATH.name} рядом с ноутбуком")
assert hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest() == (
    "4be8330d0d43324cd71f1131fd51b1c1627b284ccb0d24b7bfacb7d1b74cb782"
), "Контрольная сумма модели не совпала"

options = ort.SessionOptions()
options.intra_op_num_threads = 4
options.inter_op_num_threads = 1
session = ort.InferenceSession(
    str(MODEL_PATH), sess_options=options, providers=["CPUExecutionProvider"]
)
input_name = session.get_inputs()[0].name
print("Дообученная модель загружена.")

## Предсказание и файл для отправки

Я обрабатываю изображения пакетами по 64. Порядок строк берётся из sample_submission.csv. На выходе — ровно две колонки: image_id и p_180.

In [ ]:
batch_size = 64
predictions = np.empty(len(image_ids), dtype=np.float32)
started = time.monotonic()

for start in range(0, len(image_ids), batch_size):
    batch_ids = image_ids[start:start + batch_size]
    original, rotated = [], []
    for image_id in batch_ids:
        path = image_dir / f"{image_id}.png"
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            original.append(prepare(image))
            rotated.append(prepare(image.transpose(Image.Transpose.ROTATE_180)))
    model_input = np.stack(original + rotated)
    scores = session.run(None, {input_name: model_input})[0]
    n = len(batch_ids)
    p = (scores[:n, 1] + 1 - scores[n:, 1]) / 2
    predictions[start:start + n] = np.clip(p, 1e-5, 1 - 1e-5)
    if (start + batch_size) % 2048 == 0 or start + batch_size >= len(image_ids):
        done = min(start + batch_size, len(image_ids))
        print(f"{done:,}/{len(image_ids):,} · {time.monotonic() - started:.1f} с")

output_path = ROOT / "submission.csv"
with output_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["image_id", "p_180"])
    writer.writerows(
        (image_id, f"{float(p):.7f}")
        for image_id, p in zip(image_ids, predictions)
    )

assert np.isfinite(predictions).all()
assert np.all((predictions >= 0) & (predictions <= 1))
with output_path.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == ["image_id", "p_180"]
    rows = list(reader)
assert len(rows) == len(image_ids)
assert [row["image_id"] for row in rows] == image_ids
print(f"Готово: {output_path.name}, {len(rows):,} строк")

## Итог

Ноутбук создаёт `submission.csv` с колонками `image_id,p_180` в порядке `sample_submission.csv`. Подтверждённый балл одной дообученной CNN — **0,95582654** против **0,90713814** у первого решения.

Источники: [PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR), [RapidOCR](https://github.com/RapidAI/RapidOCR), [открытый набор для ориентации текста](https://github.com/PaddlePaddle/PaddleClas/blob/release/2.6/docs/zh_CN/models/PULC/PULC_textline_orientation.md).